In [1]:
import pandas as pd

file_path = "yellow_tripdata_2021-01.csv.gz"

df = pd.read_csv(file_path, nrows=100)

print(df.head())
print(df.dtypes)

FileNotFoundError: [Errno 2] No such file or directory: 'yellow_tripdata_2021-01.csv.gz'

In [2]:
import os

print(os.getcwd())
print(os.listdir())

/content
['.config', 'sample_data']


In [3]:
import pandas as pd

file_path = "/yellow_tripdata_2021-01 (1).csv.gz"

df = pd.read_csv(file_path, nrows=100)

print(df.head())
print(df.dtypes)

   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12                1   
1         1  2021-01-01 00:51:20   2021-01-01 00:52:19                1   
2         1  2021-01-01 00:43:30   2021-01-01 01:11:06                1   
3         1  2021-01-01 00:15:48   2021-01-01 00:31:01                0   
4         2  2021-01-01 00:31:49   2021-01-01 00:48:21                1   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           2.10           1                  N           142            43   
1           0.20           1                  N           238           151   
2          14.70           1                  N           132           165   
3          10.60           1                  N           138           132   
4           4.94           1                  N            68            33   

   payment_type  fare_amount  extra  mta_tax  tip_amount  tolls_amount  \


In [4]:

import pandas as pd

file_path = "/yellow_tripdata_2021-01 (1).csv.gz"

chunksize = 100_000

null_counts = None
min_values = {}
max_values = {}
total_rows = 0

for chunk in pd.read_csv(file_path, chunksize=chunksize):

    total_rows += len(chunk)

    if null_counts is None:
        null_counts = chunk.isnull().sum()
    else:
        null_counts += chunk.isnull().sum()

    for col in chunk.columns:
        col_min = chunk[col].min()
        col_max = chunk[col].max()

        if pd.notna(col_min):
            if col not in min_values or col_min < min_values[col]:
                min_values[col] = col_min

        if pd.notna(col_max):
            if col not in max_values or col_max > max_values[col]:
                max_values[col] = col_max

print("Total rows:", total_rows)


/tmp/ipykernel_2140/502421135.py:12: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(file_path, chunksize=chunksize):


TypeError: '<=' not supported between instances of 'str' and 'float'

In [5]:

import pandas as pd

file_path = "/yellow_tripdata_2021-01 (1).csv.gz"
chunksize = 100_000

null_counts = None
min_values = {}
max_values = {}
total_rows = 0
dtypes = None

for chunk in pd.read_csv(
    file_path,
    chunksize=chunksize,
    low_memory=False
):
    total_rows += len(chunk)

    if null_counts is None:
        null_counts = chunk.isnull().sum()
        dtypes = chunk.dtypes.astype(str)
    else:
        null_counts += chunk.isnull().sum()

    for col in chunk.columns:
        # Convert values to numeric where possible
        numeric_col = pd.to_numeric(
            chunk[col], errors="coerce"
        )

        # Calculate min and max only for numeric values
        if numeric_col.notna().any():
            col_min = numeric_col.min()
            col_max = numeric_col.max()

            if col not in min_values or col_min < min_values[col]:
                min_values[col] = col_min

            if col not in max_values or col_max > max_values[col]:
                max_values[col] = col_max

print("Total rows:", total_rows)


Total rows: 1369765


In [7]:

import pandas as pd

file_path = "/yellow_tripdata_2021-01 (1).csv.gz"

total_rows = 0
bad_dates = 0
zero_negative_fares = 0
zero_negative_distances = 0
zero_passengers = 0
duplicate_rows = 0

for chunk in pd.read_csv(
    file_path,
    chunksize=100_000,
    low_memory=False
):
    total_rows += len(chunk)

    # Convert pickup dates to datetime
    pickup = pd.to_datetime(
        chunk["tpep_pickup_datetime"],
        errors="coerce"
    )

    bad_dates += (
        pickup.isna()
        | (pickup < "2021-01-01")
        | (pickup >= "2021-02-01")
    ).sum()

    # Fare amount <= 0
    fare = pd.to_numeric(
        chunk["fare_amount"], errors="coerce"
    )
    zero_negative_fares += (fare <= 0).sum()

    # Trip distance <= 0
    distance = pd.to_numeric(
        chunk["trip_distance"], errors="coerce"
    )
    zero_negative_distances += (distance <= 0).sum()

    # Passenger count = 0
    passengers = pd.to_numeric(
        chunk["passenger_count"], errors="coerce"
    )
    zero_passengers += (passengers == 0).sum()

    # Exact duplicate rows within each chunk
    duplicate_rows += chunk.duplicated().sum()

print("Total rows:", total_rows)
print("Bad pickup dates:", bad_dates)
print("Zero or negative fares:", zero_negative_fares)
print("Zero or negative distances:", zero_negative_distances)
print("Passenger count = 0:", zero_passengers)
print("Duplicate rows within chunks:", duplicate_rows)


Total rows: 1369765
Bad pickup dates: 24
Zero or negative fares: 7411
Zero or negative distances: 19952
Passenger count = 0: 26726
Duplicate rows within chunks: 0


In [8]:

decisions = pd.DataFrame([
    {
        "problem": "Pickup dates outside January or invalid",
        "decision": "Drop",
        "why": "The assignment requires January 2021 data only.",
        "owner": "Data Engineer"
    },
    {
        "problem": "Fare amount <= 0",
        "decision": "Flag",
        "why": "Zero or negative fares need investigation before removal.",
        "owner": "Data Analyst / Business Owner"
    },
    {
        "problem": "Trip distance <= 0",
        "decision": "Flag",
        "why": "These trips may be invalid and need investigation.",
        "owner": "Data Analyst / Business Owner"
    },
    {
        "problem": "Passenger count = 0",
        "decision": "Flag",
        "why": "The passenger information may be missing or unreliable.",
        "owner": "Data Engineer"
    },
    {
        "problem": "Duplicate rows",
        "decision": "Drop",
        "why": "Remove confirmed exact duplicate records.",
        "owner": "Data Engineer"
    }
])

print(decisions)


                                   problem decision  \
0  Pickup dates outside January or invalid     Drop   
1                         Fare amount <= 0     Flag   
2                       Trip distance <= 0     Flag   
3                      Passenger count = 0     Flag   
4                           Duplicate rows     Drop   

                                                 why  \
0    The assignment requires January 2021 data only.   
1  Zero or negative fares need investigation befo...   
2  These trips may be invalid and need investigat...   
3  The passenger information may be missing or un...   
4          Remove confirmed exact duplicate records.   

                           owner  
0                  Data Engineer  
1  Data Analyst / Business Owner  
2  Data Analyst / Business Owner  
3                  Data Engineer  
4                  Data Engineer  


In [10]:

import pandas as pd

def clean(df):
    df = df.copy()
    rows_removed = {}

    rows_before = len(df)

    # Rule 1: Drop invalid pickup dates or dates outside January 2021
    pickup = pd.to_datetime(
        df["tpep_pickup_datetime"],
        errors="coerce"
    )

    bad_dates = (
        pickup.isna()
        | (pickup < "2021-01-01")
        | (pickup >= "2021-02-01")
    )

    rows_removed["bad_pickup_dates"] = int(bad_dates.sum())
    df = df.loc[~bad_dates].copy()

    # Rule 2: Flag zero or negative fares
    fare = pd.to_numeric(df["fare_amount"], errors="coerce")
    bad_fares = fare <= 0

    df["bad_fare_flag"] = bad_fares.fillna(False)

    # Rule 3: Flag zero or negative trip distances
    distance = pd.to_numeric(
        df["trip_distance"], errors="coerce"
    )
    bad_distances = distance <= 0

    df["bad_distance_flag"] = bad_distances.fillna(False)

    # Rule 4: Flag zero passenger counts
    passengers = pd.to_numeric(
        df["passenger_count"], errors="coerce"
    )

    df["zero_passenger_flag"] = passengers.eq(0)

    # Rule 5: Drop exact duplicate rows
    duplicates = df.duplicated()

    rows_removed["duplicate_rows"] = int(duplicates.sum())
    df = df.loc[~duplicates].copy()

    # Final report
    rows_removed["total_rows_before"] = rows_before
    rows_removed["total_rows_after"] = len(df)
    rows_removed["total_rows_removed"] = rows_before - len(df)

    return df, rows_removed


In [11]:

sample_df = pd.read_csv(
    file_path,
    nrows=100_000,
    low_memory=False
)

cleaned_df, rows_removed = clean(sample_df)

print("Rows before cleaning:", len(sample_df))
print("Rows after cleaning:", len(cleaned_df))
print("Rows removed per rule:")
print(rows_removed)

print("Cleaned data preview:")
print(cleaned_df.head())


Rows before cleaning: 100000
Rows after cleaning: 99984
Rows removed per rule:
{'bad_pickup_dates': 16, 'duplicate_rows': 0, 'total_rows_before': 100000, 'total_rows_after': 99984, 'total_rows_removed': 16}
Cleaned data preview:
   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12                1   
1         1  2021-01-01 00:51:20   2021-01-01 00:52:19                1   
2         1  2021-01-01 00:43:30   2021-01-01 01:11:06                1   
3         1  2021-01-01 00:15:48   2021-01-01 00:31:01                0   
4         2  2021-01-01 00:31:49   2021-01-01 00:48:21                1   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           2.10           1                  N           142            43   
1           0.20           1                  N           238           151   
2          14.70           1                  N           132           165   
3   